In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
df = pd.read_csv("bookscraper/books.csv")
df.head()

,Title,Category,Price,Rating,Availability,Description,UPC,Number of Reviews,Product URL
0,It's Only the Himalayas,Travel,£45.17,Two,In stock (19 available) In stock In stock In s...,"“Wherever you go, whatever you do, just . . . ...",a22124811bfa8350,0,https://books.toscrape.com/catalogue/its-only-...
1,Libertarianism for Beginners,Politics,£51.33,Two,In stock (19 available) In stock In stock In s...,Libertarianism isn't about winning elections; ...,a18a4f574854aced,0,https://books.toscrape.com/catalogue/libertari...
2,Mesaerion: The Best Science Fiction Stories 18...,Science Fiction,£37.59,One,In stock (19 available) In stock In stock In s...,"Andrew Barger, award-winning author and engine...",e30f54cea9b38190,0,https://books.toscrape.com/catalogue/mesaerion...
3,Olio,Poetry,£23.88,One,In stock (19 available) In stock In stock In s...,"Part fact, part fiction, Tyehimba Jess's much ...",feb7cc7701ecf901,0,https://books.toscrape.com/catalogue/olio_984/...
4,Our Band Could Be Your Life: Scenes from the A...,Music,£57.25,Three,In stock (19 available) In stock In stock In s...,This is the never-before-told story of the mus...,deda3e61b9514b83,0,https://books.toscrape.com/catalogue/our-band-...


In [4]:
print("Rows and Columns:", df.shape)

Rows and Columns: (100, 9)


In [6]:
total_records = len(df)

missing_values = df.isnull().sum().sum()

duplicate_upc = df["UPC"].duplicated().sum()

print("Total Scraped Records :", total_records)
print("Total Missing Values  :", missing_values)
print("Duplicate UPC Values  :", duplicate_upc)

Total Scraped Records : 100
Total Missing Values  : 0
Duplicate UPC Values  : 0


## Now the Data Preprocessing part 
which includes cleaning up extra spaces,improving the inconsistent text,removing duplicate books by UPC and handling missing descriptions.

In [7]:
text_columns = df.select_dtypes(include="object").columns

for col in text_columns:
    df[col] = df[col].str.strip()

print("Extra spaces removed successfully.")

Extra spaces removed successfully.


In [8]:
df["Category"] = df["Category"].str.title()
df["Rating"] = df["Rating"].str.title()
df["Availability"] = df["Availability"].str.strip()

print("Text formatting standardized.")

Text formatting standardized.


In [9]:
before = len(df)
df = df.drop_duplicates(subset="UPC")
after = len(df)

print("Records before removing duplicates:", before)
print("Records after removing duplicates :", after)
print("Duplicate books removed:", before - after)

Records before removing duplicates: 100
Records after removing duplicates : 100
Duplicate books removed: 0


In [10]:

df["Description"] = df["Description"].fillna("No Description Available")
print("Missing descriptions handled.")

Missing descriptions handled.


Now converting to numeric value, map rating One-Five to integers, and extracting available stock count

In [11]:
df["Price"] = df["Price"].str.replace("£", "", regex=False)
df["Price"] = df["Price"].astype(float)
df["Price"].head()

0    45.17
1    51.33
2    37.59
3    23.88
4    57.25
Name: Price, dtype: float64

In [12]:
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}
df["Rating"] = df["Rating"].map(rating_map)
df["Rating"].head()

0    2
1    2
2    1
3    1
4    3
Name: Rating, dtype: int64

In [13]:
df["Stock Count"] = (
    df["Availability"]
    .str.extract(r"\((\d+) available\)")
    .fillna(0)
    .astype(int)
)
df[["Availability", "Stock Count"]].head()

,Availability,Stock Count
0,In stock (19 available) In stock In stock In s...,19
1,In stock (19 available) In stock In stock In s...,19
2,In stock (19 available) In stock In stock In s...,19
3,In stock (19 available) In stock In stock In s...,19
4,In stock (19 available) In stock In stock In s...,19


saving the updated dataset

In [14]:
df.to_csv("bookscraper/books_cleaned.csv", index=False)

Adding three useful features 

In [15]:
vowels = ("A", "E", "I", "O", "U")
df["starts_with_vowel"] = df["Title"].str.upper().str.startswith(vowels)
df[["Title", "starts_with_vowel"]].head()

,Title,starts_with_vowel
0,It's Only the Himalayas,True
1,Libertarianism for Beginners,False
2,Mesaerion: The Best Science Fiction Stories 18...,False
3,Olio,True
4,Our Band Could Be Your Life: Scenes from the A...,True


In [21]:
#book whose stock is higher than 18
df["high_stock"] = df["Stock Count"] > 18
df[["Stock Count", "high_stock"]].head(10)

,Stock Count,high_stock
0,19,True
1,19,True
2,19,True
3,19,True
4,19,True
5,19,True
6,19,True
7,19,True
8,19,True
9,19,True


In [20]:
#to know which books have stock higher than 18
df[df["high_stock"]][["Title", "Stock Count"]].head(10)

,Title,Stock Count
0,It's Only the Himalayas,19
1,Libertarianism for Beginners,19
2,Mesaerion: The Best Science Fiction Stories 18...,19
3,Olio,19
4,Our Band Could Be Your Life: Scenes from the A...,19
5,Rip it Up and Start Again,19
6,Scott Pilgrim's Precious Little Life (Scott Pi...,19
7,Set Me Free,19
8,Shakespeare's Sonnets,19
9,"Starving Hearts (Triangular Trade Trilogy, #1)",19


In [22]:
#checking which books have price higher than average and have rating is 4 0r 5
average_price = df["Price"].mean()

df["premium_book"] = (
    (df["Price"] > average_price) &
    (df["Rating"] >= 4)
)

df[["Price", "Rating", "premium_book"]].head()

,Price,Rating,premium_book
0,45.17,2,False
1,51.33,2,False
2,37.59,1,False
3,23.88,1,False
4,57.25,3,False
